In [1]:
import os
import random
import json

In [6]:
root_dir = './experiment_clusters'
num_examples = 100
group_size = 5

In [ ]:
clusters = {}
for cluster_name in os.listdir(root_dir):

    cluster_path = os.path.join(root_dir, cluster_name)
    if not os.path.isdir(cluster_path):
        continue
    try:
        first, second = cluster_name.split('_')
        first = int(first) if first else None
        second = int(second)
    except ValueError:

        continue
 
    plans_dir = os.path.join(cluster_path, 'floorplan_reoriented')
    if not os.path.isdir(plans_dir):
        continue

    plan_ids = [os.path.splitext(f)[0] for f in os.listdir(plans_dir) if f.endswith('.png')]
    if len(plan_ids) < 4:
        continue
    clusters[(first, second)] = plan_ids

def sample_group(base_key, all_keys, same_second=False):
    base_ids = clusters[base_key]
    selected = random.sample(base_ids, group_size - 1)

    if same_second:
        candidates = [k for k in all_keys if k[1] == base_key[1] and k[0] != base_key[0]]
    else:
        candidates = [k for k in all_keys if k[0] != base_key[0]]
    if not candidates:
        return None
    other_key = random.choice(candidates)
    other_id = random.choice(clusters[other_key])

    return {
        'base_cluster': base_key,
        'other_cluster': other_key,
        'group': selected + [other_id],
        'outlier_id': other_id
    }


In [ ]:

groups_diff_first = []
groups_same_second = []
keys = list(clusters.keys())

while len(groups_diff_first) < num_examples:
    base = random.choice(keys)
    grp = sample_group(base, keys, same_second=False)
    if grp:
        groups_diff_first.append(grp)

while len(groups_same_second) < num_examples:
    base = random.choice(keys)
    grp = sample_group(base, keys, same_second=True)
    if grp:
        groups_same_second.append(grp)

# Save results separately
with open('groups_diff_first.json', 'w') as f:
    json.dump(groups_diff_first, f, indent=2)

with open('groups_same_second.json', 'w') as f:
    json.dump(groups_same_second, f, indent=2)

print(f"Generated {len(groups_diff_first)} 'diff_first' groups and {len(groups_same_second)} 'same_second' groups")




Generated 100 'diff_first' groups and 100 'same_second' groups


In [ ]:
# Configuration
root_dir = 'experiment_clusters'  # replace with your root path
num_examples = 100  # number of groups per scenario
group_size = 5
templates_dir = 'prompts'  # folder containing description_pick_diff_prompt.txt and json_pick_diff_prompt.txt
output_dir = './'  # where to save prompts and results

# Load prompt templates
with open(os.path.join(templates_dir, 'description_pick_diff_prompt.txt')) as f:
    desc_template = f.read()
with open(os.path.join(templates_dir, 'json_pick_diff_prompt.txt')) as f:
    json_template = f.read()

def build_prompt(sample, input_type='description'):
    # ordered retrieval: first group_size-1 from base, then the outlier
    base_ids = sample['group'][:-1]
    outlier = sample['outlier_id']
    items = base_ids + [outlier]
    # shuffle for prompt variability
    random.shuffle(items)

    bullets = []
    for idx, plan_id in enumerate(items, start=1):
        # determine which cluster to load from
        if plan_id == outlier:
            cluster = sample['other_cluster']
        else:
            cluster = sample['base_cluster']
        subfolder = 'human_annotation' if input_type == 'description' else 'json'
        ext = 'txt' if input_type == 'description' else 'json'
        file_dir = f"{cluster[0]}_{cluster[1]}"
        file_path = os.path.join(root_dir, file_dir, subfolder, f"{plan_id}.{ext}")

        if input_type == 'description':
            with open(file_path) as f:
                content = f.read().strip()
            entry = f"Example {idx}:\nID {plan_id}\n{content}\n"
        else:
            with open(file_path) as f:
                data = json.load(f)
            pretty = json.dumps(data, indent=2)
            entry = f"Example {idx}:\nID {plan_id}\n{pretty}\n"
        bullets.append(entry)

    prompt_text = "".join(bullets)
    if input_type == 'description':
        return desc_template.format(num_shots=group_size, num_shots_minus=group_size-1, description=prompt_text)
    else:
        return json_template.format(num_shots=group_size, num_shots_minus=group_size-1, example_prettified_json=prompt_text)

def load_json(fname):
    with open(fname) as f:
        return json.load(f)

diff_first_samples = load_json('groups_diff_first.json')
same_second_samples = load_json('groups_same_second.json')


eval_data = {
    'description': {'diff_first': [], 'same_second': []},
    'json': {'diff_first': [], 'same_second': []}
}

def generate_evals(samples, scenario):
    for sample in samples:
        for input_type in ['description', 'json']:
            prompt = build_prompt(sample, input_type=input_type)
            eval_data[input_type][scenario].append({
                'outlier_id': sample['outlier_id'],
                'prompt': prompt
            })

generate_evals(diff_first_samples, 'diff_first')
generate_evals(same_second_samples, 'same_second')

# Save prompts for downstream LLM calls
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, 'eval_prompts.json'), 'w') as f:
    json.dump(eval_data, f, indent=2)

print(f"Built prompts for description and JSON inputs, across both scenarios.")

Built prompts for description and JSON inputs, across both scenarios.
